# Problem 01: Academic Audit & Credit-Weighted CPI Performance Engine

### Design approach
- Use the required **master dictionary** `StudentID -> {"name": str, "courses": list}`.
- Store every course as the required immutable tuple `(CourseCode, Credits, LetterGrade)`.
- Convert grades with the given 10-point mapping and compute CPI as the credit-weighted average.
- Store each student's summary as a tuple and sort the summaries using the required three-level ranking rule:
  1. CPI descending
  2. Total credits descending
  3. Student name lexicographically ascending

### Complexity
Let `T` be the total number of course records and `N` the number of students.
- Building the records: **O(T)**
- Computing all CPIs: **O(T)**
- Merit-list sorting: **O(N log N)**
- Overall: **O(T + N log N)**
- Extra space: **O(T + N)** for course records, student summaries, and the sorted merit list.



In [1]:
from typing import Dict, List, Tuple, TypedDict

GRADE_POINTS: Dict[str, int] = {
    "A": 10,
    "B": 8,
    "C": 6,
    "D": 4,
    "F": 0,
}

class StudentRecord(TypedDict):
    """Store one student's name and immutable course records."""
    name: str
    courses: List[Tuple[str, int, str]]

def solve_problem_01(input_data: str) -> str:
    """Calculate student CPIs and produce the department merit list."""
    lines = [line.strip() for line in input_data.strip().splitlines()
             if line.strip()]
    index = 0

    student_count = int(lines[index])
    index += 1

    # Required master dictionary:
    # StudentID -> {"name": str, "courses": list}
    students: Dict[str, StudentRecord] = {}

    for _ in range(student_count):
        student_id, name = lines[index].split()
        index += 1

        course_count = int(lines[index])
        index += 1

        courses: List[Tuple[str, int, str]] = []

        for _ in range(course_count):
            course_code, credits, grade = lines[index].split()
            index += 1
            # Tuple is immutable, as required.
            courses.append((course_code, int(credits), grade))

        students[student_id] = {
            "name": name,
            "courses": courses,
        }

    summaries: List[Tuple[str, str, int, float]] = []

    for student_id, record in students.items():
        total_credits = sum(
            course[1] for course in record["courses"]
        )
        weighted_points = sum(
            course[1] * GRADE_POINTS[course[2]]
            for course in record["courses"]
        )

        cpi = round(weighted_points / total_credits, 2)
        summaries.append(
            (student_id, record["name"], total_credits, cpi)
        )

    # Required ranking:
    # CPI descending, credits descending, name ascending.
    merit_list = sorted(
        summaries,
        key=lambda item: (-item[3], -item[2], item[1]),
    )

    output = [
        f"{student_id} {name} {credits} {cpi:.2f}"
        for student_id, name, credits, cpi in summaries
    ]

    output.append("")
    output.append("-- Department Merit List --")

    output.extend(
        f"Rank {rank}: {name} (CPI: {cpi:.2f}, Credits: {credits})"
        for rank, (_, name, credits, cpi)
        in enumerate(merit_list, start=1)
    )

    return "\n".join(output)

def main() -> None:
    """Read standard input and print the solution."""
    import sys

    print(solve_problem_01(sys.stdin.read()))

# Verification: provided sample + boundary/tie case

sample_input_01 = """3
S101 Alice
3
CS101 4 A
MA101 4 B
HS101 2 A
S102 Bob
2
CS101 4 B
MA101 4 C
S103 Charlie
3
CS101 4 A
MA101 4 A
HS101 2 B
"""

expected_sample_01 = """S101 Alice 10 9.20
S102 Bob 8 7.00
S103 Charlie 10 9.60

-- Department Merit List --
Rank 1: Charlie (CPI: 9.60, Credits: 10)
Rank 2: Alice (CPI: 9.20, Credits: 10)
Rank 3: Bob (CPI: 7.00, Credits: 8)"""

sample_result_01 = solve_problem_01(sample_input_01)
print(sample_result_01)
assert sample_result_01 == expected_sample_01

# Boundary/tie case:
# Same CPI and same credits, so names must decide the ranking.
boundary_input_01 = """3
S1 Zara
1
C1 1 A
S2 Adam
1
C1 1 A
S3 Bob
1
C1 1 F
"""

boundary_result_01 = solve_problem_01(boundary_input_01)
print("\nBoundary test:")
print(boundary_result_01)

assert "Rank 1: Adam (CPI: 10.00, Credits: 1)" in boundary_result_01
assert "Rank 2: Zara (CPI: 10.00, Credits: 1)" in boundary_result_01
assert "Rank 3: Bob (CPI: 0.00, Credits: 1)" in boundary_result_01
print("\nProblem 01 verification passed.")



S101 Alice 10 9.20
S102 Bob 8 7.00
S103 Charlie 10 9.60

-- Department Merit List --
Rank 1: Charlie (CPI: 9.60, Credits: 10)
Rank 2: Alice (CPI: 9.20, Credits: 10)
Rank 3: Bob (CPI: 7.00, Credits: 8)

Boundary test:
S1 Zara 1 10.00
S2 Adam 1 10.00
S3 Bob 1 0.00

-- Department Merit List --
Rank 1: Adam (CPI: 10.00, Credits: 1)
Rank 2: Zara (CPI: 10.00, Credits: 1)
Rank 3: Bob (CPI: 0.00, Credits: 1)

Problem 01 verification passed.


# Problem 02: Library Circulation & Borrower Cohort Jaccard Similarity

### Design approach
- Use `catalog: Dict[int, Tuple[str, str]]`, so each book's `(Title, Author)` is an immutable tuple.
- Maintain the required `available_book_ids` and `borrowed_book_ids` sets.
- Maintain `student_books: Dict[str, Set[int]]` for each borrower's current portfolio.
- `BORROW` moves a book from available to borrowed and inserts it into the student's set.
- `RETURN` reverses that operation only when the student actually holds the book.
- `COMMON` uses set intersection and sorts the resulting IDs.
- `SIMILARITY` uses `|A ∩ B| / |A ∪ B|`, with `0.00` when both portfolios are empty.

### Complexity
Let `B` be the number of books and `Q` the number of operations.
- Catalog construction: **O(B)** average.
- `BORROW`, `RETURN`, and `INVENTORY`: **O(1)** average.
- `COMMON`: **O(min(|S1|, |S2|) + R log R)** where `R` is the number of common IDs printed.
- `SIMILARITY`: **O(min(|S1|, |S2|) + U)** average, where `U` is the union size.
- Overall is dominated by the set operations and output sorting across the `Q` queries.
- Extra space: **O(B + L)**, where `L` is the number of active student-book holdings.



In [2]:
from typing import Dict, List, Set, Tuple


def solve_problem_02(input_data: str) -> str:
    """Process library borrowing, returns, common books, and similarity."""
    lines = [line.strip() for line in input_data.strip().splitlines()
             if line.strip()]
    index = 0

    book_count = int(lines[index])
    index += 1

    # Required catalog:
    # BookID -> (Title, Author)
    catalog: Dict[int, Tuple[str, str]] = {}
    available_book_ids: Set[int] = set()
    borrowed_book_ids: Set[int] = set()

    for _ in range(book_count):
        book_id_text, title, author = lines[index].split()
        index += 1

        book_id = int(book_id_text)
        catalog[book_id] = (title, author)
        available_book_ids.add(book_id)

    # StudentID -> currently borrowed BookIDs
    student_books: Dict[str, Set[int]] = {}

    operation_count = int(lines[index])
    index += 1

    output: List[str] = []

    for _ in range(operation_count):
        parts = lines[index].split()
        index += 1
        command = parts[0]

        if command == "BORROW":
            student_id = parts[1]
            book_id = int(parts[2])

            if book_id not in catalog:
                output.append("Book Not Found")
            elif book_id in available_book_ids:
                available_book_ids.remove(book_id)
                borrowed_book_ids.add(book_id)
                student_books.setdefault(student_id, set()).add(book_id)
                output.append("Borrow Success")
            else:
                output.append("Already Borrowed")

        elif command == "RETURN":
            student_id = parts[1]
            book_id = int(parts[2])

            student_portfolio = student_books.get(student_id, set())

            if book_id in student_portfolio:
                student_portfolio.remove(book_id)
                borrowed_book_ids.remove(book_id)
                available_book_ids.add(book_id)
                output.append("Return Success")
            else:
                output.append("Return Error")

        elif command == "COMMON":
            student_1 = parts[1]
            student_2 = parts[2]

            books_1 = student_books.get(student_1, set())
            books_2 = student_books.get(student_2, set())
            common_books = books_1 & books_2

            if common_books:
                output.append(
                    " ".join(map(str, sorted(common_books)))
                )
            else:
                output.append("None")

        elif command == "SIMILARITY":
            student_1 = parts[1]
            student_2 = parts[2]

            books_1 = student_books.get(student_1, set())
            books_2 = student_books.get(student_2, set())

            intersection = books_1 & books_2
            union = books_1 | books_2

            if not union:
                similarity = 0.00
            else:
                similarity = len(intersection) / len(union)

            output.append(f"Similarity: {similarity:.2f}")

        elif command == "INVENTORY":
            output.append(
                f"Available: {len(available_book_ids)}, "
                f"Borrowed: {len(borrowed_book_ids)}"
            )

    return "\n".join(output)


def main() -> None:
    """Read standard input and print the solution."""
    import sys

    print(solve_problem_02(sys.stdin.read()))


# Verification: provided sample + boundary cases

sample_input_02 = """4
101 Algorithms Cormen
102 PythonCookbook Mark
103 DatabaseSystems Korth
104 OperatingSystems Silberschatz
7
BORROW S01 101
BORROW S01 102
BORROW S02 102
BORROW S02 103
COMMON S01 S02
SIMILARITY S01 S02
INVENTORY
"""

expected_sample_02 = """Borrow Success
Borrow Success
Already Borrowed
Borrow Success
None
Similarity: 0.00
Available: 1, Borrowed: 3"""

sample_result_02 = solve_problem_02(sample_input_02)
print(sample_result_02)
assert sample_result_02 == expected_sample_02

# Boundary cases:
# 1. Non-existent book
# 2. Return by a student who does not hold the book
# 3. Both portfolios empty -> similarity must be 0.00
# 4. Successful return makes the book available again
boundary_input_02 = """2
1 A1 Auth1
2 A2 Auth2
8
BORROW S01 99
RETURN S01 1
SIMILARITY S01 S02
BORROW S01 1
RETURN S01 1
BORROW S02 1
SIMILARITY S01 S02
INVENTORY
"""

boundary_result_02 = solve_problem_02(boundary_input_02)
print("\nBoundary test:")
print(boundary_result_02)

expected_boundary_02 = """Book Not Found
Return Error
Similarity: 0.00
Borrow Success
Return Success
Borrow Success
Similarity: 0.00
Available: 1, Borrowed: 1"""

assert boundary_result_02 == expected_boundary_02
print("\nProblem 02 verification passed.")


Borrow Success
Borrow Success
Already Borrowed
Borrow Success
None
Similarity: 0.00
Available: 1, Borrowed: 3

Boundary test:
Book Not Found
Return Error
Similarity: 0.00
Borrow Success
Return Success
Borrow Success
Similarity: 0.00
Available: 1, Borrowed: 1

Problem 02 verification passed.


# Problem 03: Warehouse Inventory, Tag Matching & Atomic Order Processing

### Design approach
- Use the required product catalog dictionary:
  `ProductID -> {"name": str, "price": float, "stock": int, "tags": set}`.
- Maintain `out_of_stock_ids` as a set for O(1)-average membership tracking of zero-stock products.
- Represent every incoming order as a **list of tuples** `(ProductID, RequestedQty)`.
- For an order, first aggregate repeated product IDs and validate **every** requested item before changing any stock. Therefore the transaction is atomic: either every item succeeds or nothing changes.
- `FILTER_AND` uses `issubset`; `FILTER_OR` uses set intersection.
- A successful order updates stock and inserts products that reach zero into `out_of_stock_ids`. Restocking removes them from that set.

### Complexity
Let `P` be the number of products, `Q` the number of operations, and `K` the number of items in an order.
- `FILTER_AND` / `FILTER_OR`: **O(P · T + R log R)**, where `T` is the number of queried tags and `R` is the number of matching products.
- `ORDER`: **O(K)** average, including validation and stock updates.
- `RESTOCK`: **O(1)** average.
- Overall worst-case across operations is dominated by the filtering operations.
- Extra space: **O(P + total stored tags + K)**.



In [5]:
from typing import Dict, List, Set, Tuple, TypedDict


class ProductRecord(TypedDict):
    """Store the required product attributes."""
    name: str
    price: float
    stock: int
    tags: Set[str]


def solve_problem_03(input_data: str) -> str:
    """Process product filters, atomic orders, and restocking."""
    lines = [line.strip() for line in input_data.strip().splitlines()
             if line.strip()]
    index = 0

    product_count = int(lines[index])
    index += 1

    # Required catalog:
    # ProductID -> {"name", "price", "stock", "tags"}
    catalog: Dict[int, ProductRecord] = {}
    out_of_stock_ids: Set[int] = set()

    for _ in range(product_count):
        parts = lines[index].split()
        index += 1

        product_id = int(parts[0])
        name = parts[1]
        price = float(parts[2])
        stock = int(parts[3])
        tag_count = int(parts[4])
        tags = set(parts[5:5 + tag_count])

        catalog[product_id] = {
            "name": name,
            "price": price,
            "stock": stock,
            "tags": tags,
        }

        if stock == 0:
            out_of_stock_ids.add(product_id)

    operation_count = int(lines[index])
    index += 1
    output: List[str] = []

    for _ in range(operation_count):
        parts = lines[index].split()
        index += 1
        command = parts[0]

        if command == "FILTER_AND":
            query_tags = set(parts[1:])
            matches = [
                product_id
                for product_id, product in catalog.items()
                if query_tags.issubset(product["tags"])
            ]

            output.append(
                " ".join(map(str, sorted(matches)))
                if matches
                else "No Match"
            )

        elif command == "FILTER_OR":
            query_tags = set(parts[1:])
            matches = [
                product_id
                for product_id, product in catalog.items()
                if product["tags"] & query_tags
            ]

            output.append(
                " ".join(map(str, sorted(matches)))
                if matches
                else "No Match"
            )

        elif command == "ORDER":
            customer_id = parts[1]
            item_count = int(parts[2])

            requested_items: List[Tuple[int, int]] = []
            cursor = 3

            for _ in range(item_count):
                product_id = int(parts[cursor])
                quantity = int(parts[cursor + 1])
                requested_items.append((product_id, quantity))
                cursor += 2

            # Aggregate duplicate product IDs so the total requested
            # quantity is checked before any stock is changed.
            requested_totals: Dict[int, int] = {}

            for product_id, quantity in requested_items:
                requested_totals[product_id] = (
                    requested_totals.get(product_id, 0) + quantity
                )

            valid = True
            total_cost = 0.0

            # Phase 1: validation only. No stock is mutated here.
            for product_id, quantity in requested_totals.items():
                if product_id not in catalog:
                    valid = False
                    break

                if (
                    product_id in out_of_stock_ids
                    or catalog[product_id]["stock"] < quantity
                ):
                    valid = False
                    break

                total_cost += (
                    catalog[product_id]["price"] * quantity
                )

            if not valid:
                output.append(
                    f"Order {customer_id} Rejected: Insufficient Stock"
                )
                continue

            # Phase 2: commit. This executes only after all items pass.
            for product_id, quantity in requested_totals.items():
                catalog[product_id]["stock"] -= quantity

                if catalog[product_id]["stock"] == 0:
                    out_of_stock_ids.add(product_id)

            output.append(
                f"Order {customer_id} Approved: ${total_cost:.2f}"
            )

        elif command == "RESTOCK":
            product_id = int(parts[1])
            quantity = int(parts[2])

            catalog[product_id]["stock"] += quantity
            out_of_stock_ids.discard(product_id)

            output.append("Restock Success")

    return "\n".join(output)


def main() -> None:
    """Read standard input and print the solution."""
    import sys

    print(solve_problem_03(sys.stdin.read()))


# Verification: provided sample + atomicity boundary case

sample_input_03 = """3
101 Headphones 50.00 10 3 audio tech wireless
102 Earphones 20.00 5 2 audio wired
103 Mouse 25.00 2 2 tech wireless
4
FILTER_AND tech wireless
ORDER C101 2 101 2 103 5
ORDER C101 2 101 2 103 2
RESTOCK 103 10
"""

expected_sample_03 = """101 103
Order C101 Rejected: Insufficient Stock
Order C101 Approved: $150.00
Restock Success"""

sample_result_03 = solve_problem_03(sample_input_03)
print(sample_result_03)
assert sample_result_03 == expected_sample_03

# Boundary/atomicity test:
# The first order requests 101 twice (3 + 2 = 5 units), but only 4 exist.
# The whole order must be rejected and stock must remain unchanged.
boundary_input_03 = """2
101 Keyboard 10.00 4 1 tech
102 Mouse 20.00 1 1 wireless
4
ORDER C9 2 101 3 101 2
ORDER C9 1 101 4
FILTER_OR audio
RESTOCK 101 1
"""

boundary_result_03 = solve_problem_03(boundary_input_03)
print("\nBoundary test:")
print(boundary_result_03)

expected_boundary_03 = """Order C9 Rejected: Insufficient Stock
Order C9 Approved: $40.00
No Match
Restock Success"""

assert boundary_result_03 == expected_boundary_03
print("\nProblem 03 verification passed.")


101 103
Order C101 Rejected: Insufficient Stock
Order C101 Approved: $150.00
Restock Success

Boundary test:
Order C9 Rejected: Insufficient Stock
Order C9 Approved: $40.00
No Match
Restock Success

Problem 03 verification passed.


# Problem 04: Inverted Index & Boolean Information Retrieval Engine

### Design approach
- Convert each document's text to lowercase and remove punctuation before tokenization.
- Build the required inverted index `Word -> Set[DocumentID]`.
- Maintain the universe set of active document IDs.
- Resolve:
  - `AND` with set intersection
  - `OR` with set union
  - `DIFF` with set difference
- Sort matching document IDs in ascending order before printing.

### Complexity
Let `W` be the total number of words across all documents, and let `R` be the number of document IDs returned by a query.
- Index construction: **O(W)** average for dictionary/set insertions.
- `AND`, `OR`, and `DIFF`: proportional to the relevant posting-set sizes on average.
- Sorting query results: **O(R log R)**.
- Extra space: **O(W + D)** in the worst case for the inverted index and document universe.


In [6]:
import string
from typing import Dict, List, Set


def sanitize_text(text: str) -> List[str]:
    """Lowercase text, strip punctuation, and return its words."""
    punctuation_table = str.maketrans("", "", string.punctuation)
    cleaned_text = text.lower().translate(punctuation_table)
    return cleaned_text.split()


def solve_problem_04(input_data: str) -> str:
    """Build an inverted index and execute Boolean search queries."""
    lines = input_data.strip().splitlines()
    index = 0

    document_count = int(lines[index].strip())
    index += 1

    inverted_index: Dict[str, Set[int]] = {}
    all_document_ids: Set[int] = set()

    for _ in range(document_count):
        document_id_text, document_text = lines[index].strip().split(
            maxsplit=1
        )
        index += 1

        document_id = int(document_id_text)
        all_document_ids.add(document_id)

        # A set prevents repeated occurrences of the same word in one
        # document from causing duplicate document IDs.
        unique_words = set(sanitize_text(document_text))

        for word in unique_words:
            inverted_index.setdefault(word, set()).add(document_id)

    query_count = int(lines[index].strip())
    index += 1

    output: List[str] = []

    for _ in range(query_count):
        command, term_1, term_2 = lines[index].strip().split()
        index += 1

        documents_1 = inverted_index.get(term_1, set())
        documents_2 = inverted_index.get(term_2, set())

        if command == "AND":
            result = documents_1 & documents_2
        elif command == "OR":
            result = documents_1 | documents_2
        else:  # DIFF
            result = documents_1 - documents_2

        if result:
            output.append(" ".join(map(str, sorted(result))))
        else:
            output.append("No Documents Found")

    return "\n".join(output)


def main() -> None:
    """Read standard input and print the solution."""
    import sys

    print(solve_problem_04(sys.stdin.read()))


# Verification: provided sample + missing-term boundary cases

sample_input_04 = """3
1 python data structures and algorithms
2 python machine learning and neural networks
3 database management systems and sql queries
3
AND python algorithms
OR machine sql
DIFF python machine
"""

expected_sample_04 = """1
2 3
1"""

sample_result_04 = solve_problem_04(sample_input_04)
print(sample_result_04)
assert sample_result_04 == expected_sample_04

boundary_input_04 = """2
1 Python, DATA! data.
2 SQL database.
4
AND python missing
OR python missing
DIFF missing python
AND sql database
"""

boundary_result_04 = solve_problem_04(boundary_input_04)
print("\nBoundary test:")
print(boundary_result_04)

expected_boundary_04 = """No Documents Found
1
No Documents Found
2"""

assert boundary_result_04 == expected_boundary_04
print("\nProblem 04 verification passed.")


1
2 3
1

Boundary test:
No Documents Found
1
No Documents Found
2

Problem 04 verification passed.


# Problem 05: Continuous Activity Streak & Anomaly Cluster Detection

### Design approach
- Maintain `user_logs: Dict[str, List[int]]`, exactly as required.
- For each user, convert timestamps to a set for longest-consecutive-streak detection.
- A timestamp `x` starts a streak only when `x - 1` is absent. Then count `x + 1`, `x + 2`, ... while present.
- This avoids sorting timestamps and gives the required linear-time streak algorithm under the assignment's hash-set model.
- Build frequency buckets keyed by login count, then scan counts from largest to smallest. Within the same count, rank by longest streak descending and username ascending.
- Print every user alphabetically before printing the Top-K ranking.

### Complexity
Let `M` be the number of login events and `U` the number of unique users.
- Building the user dictionary: **O(M)** average.
- Longest streak calculation across all users: **O(M)** average, because timestamps are stored in sets and each timestamp is processed as part of at most the required streak scans.
- Bucket construction: **O(U)**.
- Ranking uses the required bucket structure; tie groups are ordered by streak and username.
- Extra space: **O(M + U)**.


In [7]:
from typing import Dict, List, Tuple


def longest_streak(timestamps: List[int]) -> int:
    """Return the longest consecutive timestamp streak in O(n) average time."""
    timestamp_set = set(timestamps)
    best_streak = 0

    for timestamp in timestamp_set:
        # Only the beginning of a sequence starts a forward scan.
        if timestamp - 1 not in timestamp_set:
            current = timestamp
            current_streak = 1

            while current + 1 in timestamp_set:
                current += 1
                current_streak += 1

            best_streak = max(best_streak, current_streak)

    return best_streak


def solve_problem_05(input_data: str) -> str:
    """Compute user streaks and rank the Top-K active users."""
    lines = [line.strip() for line in input_data.strip().splitlines()
             if line.strip()]

    event_count, k = map(int, lines[0].split())

    # Required user activity index:
    # Username -> List of timestamp seconds
    user_logs: Dict[str, List[int]] = {}

    for line in lines[1:event_count + 1]:
        username, timestamp_text = line.split()
        user_logs.setdefault(username, []).append(int(timestamp_text))

    summaries: List[Tuple[str, int, int]] = []

    # Required alphabetical order for the first report.
    for username in sorted(user_logs):
        login_count = len(user_logs[username])
        streak = longest_streak(user_logs[username])
        summaries.append((username, login_count, streak))

    # Bucket sorting by login frequency.
    frequency_buckets: Dict[int, List[Tuple[str, int]]] = {}

    for username, login_count, streak in summaries:
        frequency_buckets.setdefault(login_count, []).append(
            (username, streak)
        )

    top_users: List[Tuple[str, int, int]] = []

    for login_count in sorted(frequency_buckets, reverse=True):
        # Tie-breaking within the same frequency:
        # longest streak descending, username ascending.
        candidates = sorted(
            frequency_buckets[login_count],
            key=lambda item: (-item[1], item[0]),
        )

        for username, streak in candidates:
            top_users.append((username, login_count, streak))
            if len(top_users) == k:
                break

        if len(top_users) == k:
            break

    output = [
        f"User: {username}, Logins: {login_count}, "
        f"LongestStreak: {streak}"
        for username, login_count, streak in summaries
    ]

    output.append("")
    output.append(f"-- Top {k} High-Activity Users --")

    output.extend(
        f"Rank {rank}: {username} ({login_count} logins, "
        f"streak: {streak})"
        for rank, (username, login_count, streak)
        in enumerate(top_users, start=1)
    )

    return "\n".join(output)


def main() -> None:
    """Read standard input and print the solution."""
    import sys

    print(solve_problem_05(sys.stdin.read()))


# Verification: provided sample + tie-breaking boundary case

sample_input_05 = """8 2
alice 100
bob 5
alice 4
alice 200
alice 102
alice 101
bob 6
bob 7
"""

expected_sample_05 = """User: alice, Logins: 5, LongestStreak: 3
User: bob, Logins: 3, LongestStreak: 3

-- Top 2 High-Activity Users --
Rank 1: alice (5 logins, streak: 3)
Rank 2: bob (3 logins, streak: 3)"""

sample_result_05 = solve_problem_05(sample_input_05)
print(sample_result_05)
assert sample_result_05 == expected_sample_05

# Boundary/tie case:
# All users have the same login count. The longest streak decides first;
# username decides when both login count and streak are equal.
boundary_input_05 = """6 3
zoe 10
zoe 11
adam 20
adam 30
mike 40
mike 41
"""

boundary_result_05 = solve_problem_05(boundary_input_05)
print("\nBoundary test:")
print(boundary_result_05)

# zoe and mike have streak 2; adam has streak 1.
# zoe and mike tie on both count and streak, so "mike" comes before "zoe".
assert "Rank 1: mike (2 logins, streak: 2)" in boundary_result_05
assert "Rank 2: zoe (2 logins, streak: 2)" in boundary_result_05
assert "Rank 3: adam (2 logins, streak: 1)" in boundary_result_05
print("\nProblem 05 verification passed.")


User: alice, Logins: 5, LongestStreak: 3
User: bob, Logins: 3, LongestStreak: 3

-- Top 2 High-Activity Users --
Rank 1: alice (5 logins, streak: 3)
Rank 2: bob (3 logins, streak: 3)

Boundary test:
User: adam, Logins: 2, LongestStreak: 1
User: mike, Logins: 2, LongestStreak: 2
User: zoe, Logins: 2, LongestStreak: 2

-- Top 3 High-Activity Users --
Rank 1: mike (2 logins, streak: 2)
Rank 2: zoe (2 logins, streak: 2)
Rank 3: adam (2 logins, streak: 1)

Problem 05 verification passed.
